# Apply Gold Governance

Apply Unity Catalog access for the three healthcare roles:

- `healthcare_admins`
- `healthcare_hospital_operations`
- `healthcare_researchers`

Users receive access to the governed Gold consumption views only.

Role-specific row filtering and masking are implemented inside the views using `is_account_group_member()`.

In [0]:
# Governance configuration

GOLD_CATALOG = "cdr_gold"
GOLD_SCHEMA = "analytics_mimic_iv"
FULL_SCHEMA = f"{GOLD_CATALOG}.{GOLD_SCHEMA}"

HEALTHCARE_GROUPS = [
    "healthcare_admins",
    "healthcare_hospital_operations",
    "healthcare_researchers"
]

GOVERNED_VIEWS = [
    "admission_journey",
    "icu_clinical_monitoring",
    "hospital_operational_activity"
]

print(f"Gold schema: {FULL_SCHEMA}")
print(f"Healthcare groups: {len(HEALTHCARE_GROUPS)}")
print(f"Governed views: {len(GOVERNED_VIEWS)}")

In [0]:
# Confirm the governed views exist before applying permissions

missing_views = []

for view in GOVERNED_VIEWS:
    full_name = f"{FULL_SCHEMA}.{view}"

    if not spark.catalog.tableExists(full_name):
        missing_views.append(full_name)

if missing_views:
    raise RuntimeError(
        "Missing governed views:\n" +
        "\n".join(missing_views)
    )

print("Governed views found:")
for view in GOVERNED_VIEWS:
    print(f"  {FULL_SCHEMA}.{view}")

In [0]:
# Grant access to the Gold catalog and schema

for group in HEALTHCARE_GROUPS:
    spark.sql(f"""
        GRANT USE CATALOG
        ON CATALOG {GOLD_CATALOG}
        TO `{group}`
    """)

    spark.sql(f"""
        GRANT USE SCHEMA
        ON SCHEMA {FULL_SCHEMA}
        TO `{group}`
    """)

    print(f"Catalog/schema access granted: {group}")

In [0]:
# Grant SELECT on governed consumption views only

for group in HEALTHCARE_GROUPS:
    for view in GOVERNED_VIEWS:
        full_name = f"{FULL_SCHEMA}.{view}"

        spark.sql(f"""
            GRANT SELECT
            ON VIEW {full_name}
            TO `{group}`
        """)

    print(f"View access granted: {group}")

print("Gold governance grants applied successfully.")

In [0]:
# Verify SELECT permissions on each governed view

for view in GOVERNED_VIEWS:
    full_name = f"{FULL_SCHEMA}.{view}"

    print(f"\n--- {view} ---")

    display(
        spark.sql(
            f"SHOW GRANTS ON VIEW {full_name}"
        ).filter(
            "Principal IN "
            "('healthcare_admins', "
            "'healthcare_hospital_operations', "
            "'healthcare_researchers')"
        )
    )

In [0]:
# Review catalog/schema grants for overly broad access

print("Catalog grants")
display(
    spark.sql(
        f"SHOW GRANTS ON CATALOG {GOLD_CATALOG}"
    ).filter(
        "Principal IN "
        "('healthcare_admins', "
        "'healthcare_hospital_operations', "
        "'healthcare_researchers')"
    )
)

print("Schema grants")
display(
    spark.sql(
        f"SHOW GRANTS ON SCHEMA {FULL_SCHEMA}"
    ).filter(
        "Principal IN "
        "('healthcare_admins', "
        "'healthcare_hospital_operations', "
        "'healthcare_researchers')"
    )
)